In [ ]:
import time
from pathlib import Path

CODE = Path.cwd()
ROOT = CODE.parent
DATA = ROOT / 'data'               

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from mg_model import modular_MG_model as mod
from specifications import SPEC
from architechture import AC_Arch, DC_Arch
from mg_model import plot_functions as pf

In [ ]:
# Load Data 
spot_df = pd.read_csv(DATA / 'spot_df.csv', parse_dates = ['HourUTC'])
price_15min = spot_df.set_index('HourUTC')['SpotPriceEUR'].resample('15min').ffill()

pv_raw = pd.read_csv(DATA / 'pv_raw.csv', index_col = 'time', parse_dates = ['time'])
pv_15min = (pv_raw['P'] / 1e6).resample('15min').interpolate('linear')

# Day-ahead scheduling test one day
load_day  = np.full(SPEC.N_T, SPEC.LOAD_MW) 

Run for 20 Random days

In [ ]:
rng = np.random.default_rng(0)
all_days = pd.date_range('2025-01-02', '2025-09-29', freq = 'D')
days = pd.DatetimeIndex(rng.choice(all_days, size = 20, replace = False)).sort_values().strftime('%Y-%m-%d')

results = []
for arch, name in [(AC_Arch, 'AC'), (DC_Arch, 'DC')]:
    for d in days:
        sc = mod.Scenario(n_t = SPEC.N_T, dt = SPEC.DT, date = d,
                          data = {'grid': {'price': price_15min[d].values}, 'PV': {'production': pv_15min[d].values}, 'bess': {'soc_init': 0.5}, 'load': {'demand': load_day}})
        results.append(mod.build_and_solve(arch, sc, name = name, verbose = False))

print('\nsummary'); pf.summary_table(results, show = True)
print('\nlosses by stage [MWh]'); pf.loss_table(results, show = True)
print('\nstage share of own total [%]'); pf.stage_share_table(results, show = True)

for f in (pf.plot_cost, pf.plot_grid_import, pf.plot_losses, pf.plot_efficiency, pf.plot_bess_utilization, pf.plot_loss_breakdown):
    f(results)

d = days[9]
pf.plot_dispatch([r for r in results if r['date'] == d], index = price_15min[d].index)
plt.show()
